In [1]:
!pip install h5netcdf
!pip install pyhdf
!pip install cmocean
!pip install earthpy

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [9]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import h5netcdf
import rioxarray as rxr
import h5py
import rasterio
import netCDF4
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.axes as ax
import dask.array as da
import cartopy.feature as cfeature
import datetime as dt
import os
import glob
import scipy.ndimage as ndimage
import warnings
import pyhdf
import matplotlib.colors as mcolors 
import matplotlib.patches as mpatches 
import warnings
import h5py
import pyproj
import rioxarray
import geopandas as gpd 
import matplotlib.lines as mlines
import dbfread
import pymannkendall
import osmnx as ox
import numpy.ma as ma
import numba
import pymannkendall as mk
import folium
import imageio
import base64

from simpledbf import Dbf5
from rasterio.mask import mask
from tqdm import tqdm
from io import BytesIO
from rasterio.plot import show
from folium.raster_layers import ImageOverlay
from libpysal.weights import lat2W
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from matplotlib.ticker import MultipleLocator
from sklearn.linear_model import LinearRegression
from scipy.ndimage import uniform_filter
from pyproj import Transformer, CRS
from folium import LayerControl
from branca.element import Template, MacroElement
from rasterio.transform import from_bounds
from rasterio.warp import reproject, calculate_default_transform
from rasterio.enums import Resampling
from rasterio.features import rasterize
from collections import defaultdict
from dask.diagnostics import ProgressBar
from matplotlib.colors import ListedColormap, BoundaryNorm, TwoSlopeNorm
from matplotlib.ticker import MaxNLocator, MultipleLocator
from datetime import datetime, timedelta
from scipy.ndimage import gaussian_filter
from pyhdf.SD import SD, SDC
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch
from rasterio.warp import transform
from osgeo import gdal, osr
from matplotlib.patches import Rectangle
from mpl_toolkits.basemap import Basemap
from netCDF4 import Dataset
from cartopy.io.shapereader import Reader
from rasterio.warp import transform
from rasterio.features import geometry_mask
from scipy import interpolate
from scipy.signal import savgol_filter
from rasterio import features
from scipy.ndimage import gaussian_filter, generic_filter
from affine import Affine
from datetime import datetime, timedelta
from dask import delayed, compute
from osgeo import gdal
from pymannkendall import original_test

warnings.filterwarnings('ignore')

In [11]:
out_html = "HAPI_HD_daynight_final.html"
dst_crs = "EPSG:4326"
pixel_size_deg = 0.00005 

src_crs_str = "+proj=sinu +R=6371007.181 +nadgrids=@null +wktext"

hapi_colors = {
    1: mcolors.to_rgba("#6F8FBF", 0.35),
    2: mcolors.to_rgba("#B7C9DD", 0.40),
    3: mcolors.to_rgba("#F2EDB4", 0.45),
    4: mcolors.to_rgba("#D8C3A5", 0.60),
    5: mcolors.to_rgba("#E74C3C", 0.60),
}

priority_labels = {1:"Minimal",2:"Low",3:"Moderate",4:"High",5:"Critical"}
palette = {k: tuple(int(255*c) for c in v) for k,v in hapi_colors.items()}

ds = xr.open_dataset("D:/UHI_Papers_Analysis/Data/UHI_Results/HAPI/HAPI_HD_daynight.nc")
hapi_dict = {
    "Day": ds["HAPI_Day"],
    "Night": ds["HAPI_Night"]
}

urban = gpd.read_file("D:/UHI_Papers_Analysis/Data/Notebooks/SHP_Sethi/IndiaUrban.shp")
urban = urban[urban["CityName"]=="National Capital Territory"].to_crs(dst_crs)
center = urban.geometry.centroid.iloc[0]
center_latlon = [center.y, center.x]

additional_locations = {
    "Ashok \nVihar": (77.17270493379824, 28.692667766699582), 
    "  Preet Vihar": (77.29247783727455, 28.636319192240105),
    "Rohini": (77.08128940274396, 28.737767292628188),
    "Gurugram": (77.0166, 28.4595),
    "Faridabad": (77.2778, 28.4199),
    "Najafgarh": (76.98752148653183, 28.61116992585332),
    "Lajpat Nagar": (77.2433, 28.5677),
    "Greater \nNoida": (77.4940, 28.4744),
    "Alipur": (77.14543845579491, 28.79716249119533),
    "IGI \nAirport": (77.0881553144882, 28.54427388347985),                                    
    "Ghaziabad": (77.4000, 28.6692),
    "Sanjay \nVan": (77.17866558563006, 28.535619671685403),
    "Shahdara": (77.2903942106499, 28.690191137693517),
    "Nangoli": (77.06147455794951, 28.68049370237828),
    "Bahadurgarh": (76.9314, 28.6814),
    "Aravalli \n  Park": (77.11135308488846, 28.483318778580404),
    "\nRidge\nForest":(77.16928857022805, 28.6175924383604),
    "\nSarurpur\nIndustrial\nArea": (77.27144603795585, 28.352669942215773) 
}
bold_labels = {"Gurugram","Faridabad"}

def reproject_to_latlon(src_arr, src_x, src_y):
    """Reproject raster array to EPSG:4326"""
    left, right = src_x.min(), src_x.max()
    bottom, top = src_y.min(), src_y.max()
    h, w = src_arr.shape
    src_transform = from_bounds(left, bottom, right, top, w, h)
    dst_transform, dst_w, dst_h = calculate_default_transform(src_crs_str, dst_crs, w, h, left, bottom, right, top, dst_resolution=(pixel_size_deg, pixel_size_deg))
    dst_arr = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
    reproject(src_arr, dst_arr,src_transform=src_transform,src_crs=src_crs_str,dst_transform=dst_transform,dst_crs=dst_crs,resampling=Resampling.nearest)
    return dst_arr, dst_transform

def da_to_rgba(arr):                                      # Convert numeric HAPI array to RGBA
    h, w = arr.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    for cls, col in palette.items():
        rgba[arr==cls] = col
    rgba[np.isnan(arr)] = (0,0,0,0)
    return rgba

def overlay_from_array(dst_arr, dst_transform):            # Convert array to base64 image URL for Folium overlay
    rgba = da_to_rgba(dst_arr)
    buffer = BytesIO()
    imageio.imwrite(buffer, rgba, format='png')
    encoded = base64.b64encode(buffer.getvalue()).decode()
    return f"data:image/png;base64,{encoded}"

m = folium.Map(location=center_latlon, zoom_start=10, tiles=None, prefer_canvas=True)
folium.TileLayer(tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",attr="CartoDB",control=False).add_to(m)

for label, da in hapi_dict.items():
    dst_arr, dst_transform = reproject_to_latlon(da.values, da.x.values, da.y.values)

    mask = rasterize([(g,1) for g in urban.geometry], out_shape=dst_arr.shape, transform=dst_transform, fill=0)
    dst_arr[mask==0] = np.nan

    left = dst_transform.c
    top = dst_transform.f
    right = left + dst_transform.a * dst_arr.shape[1]
    bottom = top + dst_transform.e * dst_arr.shape[0]
    bounds = [[bottom, left], [top, right]]

    img_url = overlay_from_array(dst_arr, dst_transform)
    fg = folium.FeatureGroup(name=f"HAPI {label}", show=(label=="Day"))
    ImageOverlay(img_url, bounds=bounds, opacity=1.0, interactive=False).add_to(fg)
    fg.add_to(m)


folium.GeoJson(urban,style_function=lambda f: {"color":"#8B0000","weight":2.8,"fillOpacity":0}).add_to(m)

for name, (lon, lat) in additional_locations.items():
    folium.CircleMarker(location=[lat, lon],radius=4.5,color="black",fill=True,fill_color="lightblue",fill_opacity=1.0,weight=1.4,tooltip=name,popup=name).add_to(m)

legend_html = """{% macro html(this, kwargs) %}
<div style="
position:fixed;
bottom:30px;
left:20px;
width:220px;
background:white;
padding:10px;
border-radius:6px;
box-shadow:2px 2px 6px rgba(0,0,0,0.3);
font-size:14px;
z-index:9999;">
<b>Heat Adaptation Priority Index</b><br><br>
"""
for k in sorted(priority_labels.keys()):
    legend_html += f'<i style="background:{mcolors.to_hex(hapi_colors[k][:3])};width:14px;height:14px;display:inline-block;margin-right:6px;"></i> {priority_labels[k]}<br>'
legend_html += "</div>{% endmacro %}"

legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

LayerControl(collapsed=False).add_to(m)

m.save(out_html)
print(f"Interactive HAPI map saved as: {out_html}")

Interactive HAPI map saved as: HAPI_HD_daynight_final.html
